In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
markers_df = pd.read_csv("lifu_markers_1_2back_-1580742246.csv")
calibration_complete =markers_df[markers_df["marker"] == "START_EXPERIMENT_RECEIVED"].iloc[-1]
calibration_complete = calibration_complete["LSL_timestamp"]
markers_df_on = markers_df[markers_df["marker"] =="LIFU_ON"]
markers_df

,Time,marker,LSL_timestamp
0,3.526383,collecting_baseline,812420.934331
1,3.573249,collecting_baseline,812420.981196
2,3.605256,collecting_baseline,812421.013203
3,3.660293,collecting_baseline,812421.068240
4,3.691872,collecting_baseline,812421.099819
...,...,...,...
201,49.694045,START_EXPERIMENT_RECEIVED,812467.101992
202,53.007537,LIFU_ON,812470.415484
203,58.009965,LIFU_OFF,812475.417912
204,100.619525,LIFU_ON,812518.027473


In [3]:
calibration_complete  # remove w actual marker time

812467.101992

In [4]:
eeg_df = pd.read_csv("scope_eeg_2back_-1580742246.csv")
eeg_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,Ch07
0,812417.875736,62061.082031,8796.252930,7.737406e+07,1.090938e+07,2.609900e+06,3.385415e+06,-79.142563
1,812417.876299,58091.617188,8997.518555,8.095535e+07,1.252845e+07,2.997237e+06,3.385415e+06,-78.686386
2,812417.877510,56806.238281,9007.474609,8.113459e+07,1.415104e+07,3.385415e+06,3.385415e+06,-78.375694
3,812417.877961,60674.699219,8818.864258,7.777238e+07,1.570620e+07,3.757463e+06,3.385415e+06,-78.050636
4,812417.947974,62408.886719,8429.223633,7.105182e+07,1.712658e+07,4.097267e+06,3.385415e+06,-77.448647
...,...,...,...,...,...,...,...,...
28874,812533.478355,51178.175781,-1.181619,1.396223e+00,1.637804e+00,-1.632048e-01,-1.793788e-01,0.167271
28875,812533.479335,52020.675781,-0.822324,6.762159e-01,1.650593e+00,-1.601452e-01,-1.793788e-01,0.165554
28876,812533.479673,51710.062500,-0.432153,1.867560e-01,1.654201e+00,-1.592820e-01,-1.793788e-01,0.132947
28877,812534.483407,0.000000,-0.019979,3.991672e-04,1.654193e+00,-1.592841e-01,-1.793788e-01,-1.557345


In [6]:
eeg_df = eeg_df.rename(columns = {"Time": "LSL Time", "Ch01": "Raw", "Ch02": "Theta Bandpass", "Ch03": "Power", "Ch04": "Moving Average", "Ch05": "Z-score", "Ch06": "Hold", "Ch07": "Trigger Channel"})
eeg_df

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel
0,812417.875736,62061.082031,8796.252930,7.737406e+07,1.090938e+07,2.609900e+06,3.385415e+06,-79.142563
1,812417.876299,58091.617188,8997.518555,8.095535e+07,1.252845e+07,2.997237e+06,3.385415e+06,-78.686386
2,812417.877510,56806.238281,9007.474609,8.113459e+07,1.415104e+07,3.385415e+06,3.385415e+06,-78.375694
3,812417.877961,60674.699219,8818.864258,7.777238e+07,1.570620e+07,3.757463e+06,3.385415e+06,-78.050636
4,812417.947974,62408.886719,8429.223633,7.105182e+07,1.712658e+07,4.097267e+06,3.385415e+06,-77.448647
...,...,...,...,...,...,...,...,...
28874,812533.478355,51178.175781,-1.181619,1.396223e+00,1.637804e+00,-1.632048e-01,-1.793788e-01,0.167271
28875,812533.479335,52020.675781,-0.822324,6.762159e-01,1.650593e+00,-1.601452e-01,-1.793788e-01,0.165554
28876,812533.479673,51710.062500,-0.432153,1.867560e-01,1.654201e+00,-1.592820e-01,-1.793788e-01,0.132947
28877,812534.483407,0.000000,-0.019979,3.991672e-04,1.654193e+00,-1.592841e-01,-1.793788e-01,-1.557345


In [7]:
lifu_on = markers_df[markers_df["marker"]=="LIFU_ON"].copy()
lifu_on_timestamp = lifu_on["LSL_timestamp"]
lifu_on_timestamp = np.array(lifu_on_timestamp)
lifu_on_timestamp

array([812470.4154841, 812518.0274728])

In [9]:
target_ts = lifu_on_timestamp

tol = 3e-3  # tolerance for float comparison

df_matches = eeg_df[
    eeg_df["LSL Time"].apply(
        lambda x: np.any(np.abs(target_ts - x) < tol)
    )
]

df_matches

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel
13111,812470.414333,53605.679688,1.230616,1.514415,9.366163,1.685685,1.698539,0.030137
13112,812470.414719,54500.894531,1.679036,2.819162,9.419894,1.698539,1.698539,-0.258067
13113,812470.416539,54350.523438,2.077770,4.317130,9.505529,1.719026,1.698539,-0.480048
25011,812518.025033,53519.062500,1.408100,1.982746,9.037740,1.607115,1.596296,-0.031292
25012,812518.025661,53214.531250,0.834743,0.696796,8.992516,1.596296,1.596296,0.158147
25013,812518.026834,50559.593750,0.249472,0.062236,8.962255,1.589056,1.596296,0.205823


In [10]:
merged_df = eeg_df

In [11]:
buffer = []
offline_z = []
medians = []
mads = []
last_theta_val = None
last_median_val = None
last_mad_val = None

SONICATION_TIME = 5 #seconds i believe  
COOLDOWN_TIME = 15 #sonication time + cooldown time 
THETA_THRESHOLD_Z = 1.5    # z-score threshold
MU = 3.12
SIGMA =  5.31
MAD_THRESHOLD = 6       # for artifact rejection in baseline collection
INITIAL_CUTOFF = 100.0   # initial power threshold to exclude extreme artifacts
BUFFER_SIZE = 500
sonication_enabled = calibration_complete

# Ch05 = index 4
channel = "Hold"

for i in range(len(merged_df)):
    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(last_theta_val)
        medians.append(last_median_val)
        mads.append(last_theta_val)
        continue
    # Warm-up
    if len(buffer) <= 200:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        medians.append(np.nan)
        mads.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6
    

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        medians.append(np.nan)
        mads.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)
    last_theta_val = theta_val
    last_median_val = median
    last_mad_val = mad

    # Z-score
    offline_z.append(theta_val)
    medians.append(median)
    mads.append(mad)

merged_df["offline z-score"] = offline_z
merged_df["median"] = medians
merged_df['mad'] = mads




In [12]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["offline z-score"].iloc[i]
    now = merged_df["LSL Time"].iloc[i]

    if now> sonication_enabled and theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now


In [13]:
merged_df["LIFU"] = LIFU
merged_df

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU
0,812417.875736,62061.082031,8796.252930,7.737406e+07,1.090938e+07,2.609900e+06,3.385415e+06,-79.142563,NaN,NaN,NaN,0.0
1,812417.876299,58091.617188,8997.518555,8.095535e+07,1.252845e+07,2.997237e+06,3.385415e+06,-78.686386,NaN,NaN,NaN,0.0
2,812417.877510,56806.238281,9007.474609,8.113459e+07,1.415104e+07,3.385415e+06,3.385415e+06,-78.375694,NaN,NaN,NaN,0.0
3,812417.877961,60674.699219,8818.864258,7.777238e+07,1.570620e+07,3.757463e+06,3.385415e+06,-78.050636,NaN,NaN,NaN,0.0
4,812417.947974,62408.886719,8429.223633,7.105182e+07,1.712658e+07,4.097267e+06,3.385415e+06,-77.448647,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
28874,812533.478355,51178.175781,-1.181619,1.396223e+00,1.637804e+00,-1.632048e-01,-1.793788e-01,0.167271,-0.179379,-0.076593,-0.179379,0.0
28875,812533.479335,52020.675781,-0.822324,6.762159e-01,1.650593e+00,-1.601452e-01,-1.793788e-01,0.165554,-0.179379,-0.076593,-0.179379,0.0
28876,812533.479673,51710.062500,-0.432153,1.867560e-01,1.654201e+00,-1.592820e-01,-1.793788e-01,0.132947,-0.179379,-0.076593,-0.179379,0.0
28877,812534.483407,0.000000,-0.019979,3.991672e-04,1.654193e+00,-1.592841e-01,-1.793788e-01,-1.557345,-0.179379,-0.076593,-0.179379,0.0


In [14]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on_df = merged_df[mask_lifu]
lifu_on_df["LSL Time"] = lifu_on_df["LSL Time"]
lifu_on_cal = lifu_on_df[lifu_on_df["LSL Time"] > calibration_complete]
lifu_on_cal

C:\Users\jshin\AppData\Local\Temp\ipykernel_26188\4113107194.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lifu_on_df["LSL Time"] = lifu_on_df["LSL Time"]


,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU
13111,812470.414333,53605.679688,1.230616,1.514415,9.366163,1.685685,1.698539,0.030137,1.698539,-0.025595,0.308269,1.0
25011,812518.025033,53519.062500,1.408100,1.982746,9.037740,1.607115,1.596296,-0.031292,1.596296,-0.104993,0.289341,1.0


In [15]:
lifu_on_time = lifu_on_cal["LSL Time"]
lifu_on_time = np.array(lifu_on_time)
offline_lifu= lifu_on_time

In [16]:
offline_lifu # because the begin calibration didn't start yet that's why it doesn't

array([812470.4143332, 812518.0250331])

In [17]:
target_ts

array([812470.4154841, 812518.0274728])

In [18]:
calibration_complete

812467.101992

In [19]:
offline_lifu - target_ts

array([-0.0011509, -0.0024397])

In [34]:
channel_8 = eeg_df.nlargest(15, "Trigger Channel")
channel_8

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU
25072,812518.264842,49964.769531,-2.760473,7.620210,6.695447,1.046758,1.046758,895.706848,1.046758,-0.087840,1.046758,0.0
25073,812518.271759,52218.273438,-2.691713,7.245321,6.512301,1.002943,1.046758,850.900757,1.046758,-0.087840,1.046758,0.0
25071,812518.264646,51522.804688,-2.770898,7.677878,6.842854,1.082022,1.046758,768.759277,1.046758,-0.087840,1.046758,0.0
13179,812470.682607,54578.550781,-1.019117,1.038600,1.238740,-0.258675,-0.249437,768.210938,-0.249437,-0.023534,-0.249437,0.0
13180,812470.682815,54105.261719,-0.754471,0.569227,1.249882,-0.256009,-0.249437,764.963562,-0.249437,-0.023534,-0.249437,0.0
13178,812470.680838,53932.175781,-1.257144,1.580410,1.222211,-0.262629,-0.249437,716.244141,-0.249437,-0.023534,-0.249437,0.0
13181,812470.688695,53194.980469,-0.469426,0.220361,1.253330,-0.255184,-0.249437,707.175171,-0.249437,-0.023534,-0.249437,0.0
25074,812518.273679,53939.917969,-2.565803,6.583344,6.298991,0.951912,1.046758,702.242065,1.046758,-0.087840,1.046758,0.0
13177,812470.669809,53175.886719,-1.463228,2.141036,1.204182,-0.266942,-0.249437,600.917175,-0.249437,-0.023534,-0.249437,0.0
13182,812470.698705,53730.679688,-0.170993,0.029239,1.248528,-0.256333,-0.256333,597.141907,-0.256333,-0.025301,0.309972,0.0


In [35]:
channel_8 = eeg_df.nlargest(15, "Trigger Channel")
channel_8_sorted = channel_8.sort_values("LSL Time", ascending=False) # descending
MIN_SPACING = 10.0  # seconds
kept_rows = []
last_time = 0

for _, row in channel_8_sorted.iterrows():
    t = row["LSL Time"]
    if np.abs(t - last_time) >= MIN_SPACING:
        kept_rows.append(row)
        last_time = t
    if len(kept_rows) == 15:
        break
ttl_signal = pd.DataFrame(kept_rows)
ttl_signal_time = np.array(ttl_signal['LSL Time'])
ttl_signal_time = np.sort(ttl_signal_time)
target_ts- target_ttl

array([-0.2391809, -0.2164462])

In [39]:
#windowing raw eeg
raw_with08 = pd.read_csv("offline_2back_-1580742246_20260622_100458.csv")
raw = raw_with08.drop(' Ch08', axis = 1)
pre = 2
post = 5
raw_windows = []
target_ts_onset = np.array(lifu_on['Time'])


for idx, event in enumerate(target_ts_onset): 
    mask = (raw['Time'] >= (event - pre)) & (raw['Time'] <= (event + post))
    window_df = raw.loc[mask].copy()
    window_df["t_rel"] = window_df["Time"] - event 
    window_df["idx"] = idx
    raw_windows.append(window_df)
raw_windows[0]

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,Ch07,t_rel,idx
12752,51.008,54464.054688,24308.683594,116.968163,26027.324219,23364.234375,54875.492188,41796.042969,-1.999537,0
12753,51.012,54632.738281,24320.869141,417.470947,26058.009766,23530.869141,55253.773438,40616.136719,-1.995537,0
12754,51.016,53774.718750,23934.437500,-45.895580,25763.511719,23175.958984,54589.585938,41897.636719,-1.991537,0
12755,51.020,53424.242188,23850.703125,-447.988525,25681.496094,22932.962891,54047.968750,43425.968750,-1.987537,0
12756,51.024,54310.968750,24245.812500,3.862381,25974.371094,23291.375000,54724.812500,42127.875000,-1.983537,0
...,...,...,...,...,...,...,...,...,...,...
14497,57.988,53021.429688,23360.802734,-951.623901,25341.417969,22464.730469,53610.738281,42980.988281,4.980463,0
14498,57.992,53854.398438,23726.417969,-536.537292,25579.025391,22805.121094,54247.144531,41718.199219,4.984463,0
14499,57.996,54367.781250,23888.353516,-62.203411,25749.636719,23116.568359,54876.480469,40152.742188,4.988463,0
14500,58.000,53710.300781,23586.849609,-358.581543,25572.109375,22867.802734,54425.429688,41045.144531,4.992463,0


In [40]:
#windowing scope eeg
pre = 2
post = 5
theta_windows = []


for idx, event in enumerate(target_ts): 
    mask = (eeg_df['LSL Time'] >= (event - pre)) & (eeg_df['LSL Time'] <= (event + post))
    window_df = eeg_df.loc[mask].copy()
    window_df["t_rel"] = window_df["LSL Time"] - event 
    window_df["idx"] = idx
    theta_windows.append(window_df)
theta_windows[0]

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,Trigger Channel,offline z-score,median,mad,LIFU,t_rel,idx
12613,812468.417670,54424.363281,12.740400,162.317795,850.326050,202.872269,211.739838,0.065773,NaN,NaN,NaN,0.0,-1.997814,0
12614,812468.418769,53465.417969,9.911955,98.246849,808.403992,192.843063,211.739838,-0.108464,NaN,NaN,NaN,0.0,-1.996715,0
12615,812468.424920,53738.457031,6.987566,48.826080,762.991577,181.978851,211.739838,-0.298684,NaN,NaN,NaN,0.0,-1.990564,0
12616,812468.439112,54634.742188,4.000474,16.003796,715.548645,170.628860,211.739838,-0.398516,NaN,NaN,NaN,0.0,-1.976372,0
12617,812468.439519,54543.886719,0.985392,0.970998,667.551453,159.146286,211.739838,-0.320242,NaN,NaN,NaN,0.0,-1.975965,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14355,812475.400797,54125.957031,0.534731,0.285937,0.241354,-0.497284,-0.501056,1.376642,-0.501056,-0.068488,-0.501056,0.0,4.985313,0
14356,812475.403744,53166.273438,0.473984,0.224660,0.245427,-0.496309,-0.501056,38.438763,-0.501056,-0.068488,-0.501056,0.0,4.988260,0
14357,812475.404569,53280.363281,0.414445,0.171765,0.248716,-0.495522,-0.501056,22.568970,-0.501056,-0.068488,-0.501056,0.0,4.989085,0
14358,812475.404834,54214.769531,0.357619,0.127891,0.251244,-0.494918,-0.501056,-23.474102,-0.501056,-0.068488,-0.501056,0.0,4.989350,0


In [42]:
base_dir_raw = r"C:\Users\jshin\OW_closedloopLIFU\eeg_windows\raw"
base_dir_scope=r"C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope"
# MAKE SURE TO PUT IN NAME
new_folder_name = "demo1023"
new_folder_path_raw = os.path.join(base_dir_raw, new_folder_name)
new_folder_path_scope = os.path.join(base_dir_scope, new_folder_name)
os.makedirs(new_folder_path_raw, exist_ok=False) # check
os.makedirs(new_folder_path_scope, exist_ok=False) # check

for idx, event in enumerate(target_ts_onset):
    # Build a unique filename for each event
    output_path_raw = os.path.join(new_folder_path_raw, f"{idx}_{event}_eeg_raw.csv")
    output_path_scope = os.path.join(new_folder_path_scope, f"{idx}_{event}_eeg_scope.csv")
    try:
        raw_windows[idx].to_csv(output_path_raw, index=False, encoding="utf-8")
        theta_windows[idx].to_csv(output_path_scope, index=False, encoding="utf-8")
        print(f"Saved successfully to: {output_path_raw}")
        print(f"Saved successfully to: {output_path_scope}")
    except (OSError, IOError) as e:
        print(f"Error saving CSV file: {e}")


Saved successfully to: C:\Users\jshin\OW_closedloopLIFU\eeg_windows\raw\demo1023\0_53.0075366999954_eeg_raw.csv
Saved successfully to: C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope\demo1023\0_53.0075366999954_eeg_scope.csv
Saved successfully to: C:\Users\jshin\OW_closedloopLIFU\eeg_windows\raw\demo1023\1_100.61952539999038_eeg_raw.csv
Saved successfully to: C:\Users\jshin\OW_closedloopLIFU\eeg_windows\scope\demo1023\1_100.61952539999038_eeg_scope.csv
